# Baseline de popularidade

## Objetivo

Criar e avaliar um recomendador simples baseado na popularidade dos filmes nos dados de treino.

Esse modelo será a referência mínima para comparar modelos personalizados posteriores. O notebook usa split temporal, calcula o ranking sem consultar o futuro e remove filmes que cada usuário já avaliou.

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

TOP_K = 10
RELEVANCE_THRESHOLD = 4.0

## Localização e carregamento dos dados

A raiz é localizada automaticamente, portanto o notebook funciona quando aberto pela raiz do projeto ou pela pasta `notebooks/`.

In [3]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Não foi possível localizar data/raw.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

ratings = pd.read_csv(RAW_DATA_DIR / "ratings.csv").rename(
    columns={"userId": "user_id", "movieId": "movie_id"}
)
movies = pd.read_csv(RAW_DATA_DIR / "movies.csv").rename(
    columns={"movieId": "movie_id"}
)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Ratings: {len(ratings):,} | Filmes: {len(movies):,}")

Raiz do projeto: /home/mkiku/ml-practice/MovieLens_project
Ratings: 100,836 | Filmes: 9,742


## Split temporal por usuário

Para cada usuário, a última interação é reservada para teste e a penúltima para validação. Todas as interações anteriores formam o treino. Assim, o modelo sempre tenta prever o futuro usando apenas o passado.

In [4]:
def temporal_leave_two_out(
    interactions: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    ordered = interactions.assign(
        rated_at=pd.to_datetime(interactions["timestamp"], unit="s", utc=True)
    ).sort_values(["user_id", "rated_at", "movie_id"], kind="stable")

    ordered["interaction_order"] = ordered.groupby("user_id").cumcount()
    ordered["history_size"] = ordered.groupby("user_id")["movie_id"].transform("size")
    ordered["position_from_end"] = (
        ordered["history_size"] - ordered["interaction_order"]
    )

    test = ordered.loc[ordered["position_from_end"] == 1].copy()
    validation = ordered.loc[ordered["position_from_end"] == 2].copy()
    train = ordered.loc[ordered["position_from_end"] > 2].copy()
    return train, validation, test


train, validation, test = temporal_leave_two_out(ratings)

In [5]:
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "users": [
            train["user_id"].nunique(),
            validation["user_id"].nunique(),
            test["user_id"].nunique(),
        ],
        "relevant_rows": [
            int((train["rating"] >= RELEVANCE_THRESHOLD).sum()),
            int((validation["rating"] >= RELEVANCE_THRESHOLD).sum()),
            int((test["rating"] >= RELEVANCE_THRESHOLD).sum()),
        ],
    }
)
split_summary

,split,rows,users,relevant_rows
0,train,99616,610,47870
1,validation,610,610,347
2,test,610,610,363


### Validações do split

As verificações garantem que nenhuma linha foi perdida, que cada usuário aparece nos três conjuntos e que treino e validação são anteriores ao teste.

In [6]:
n_users = ratings["user_id"].nunique()

assert len(train) + len(validation) + len(test) == len(ratings)
assert train["user_id"].nunique() == n_users
assert validation["user_id"].nunique() == n_users
assert test["user_id"].nunique() == n_users
assert not set(train.index) & set(validation.index)
assert not set(train.index) & set(test.index)
assert not set(validation.index) & set(test.index)

train_max_time = train.groupby("user_id")["rated_at"].max()
validation_time = validation.set_index("user_id")["rated_at"]
test_time = test.set_index("user_id")["rated_at"]

assert (train_max_time <= validation_time).all()
assert (validation_time <= test_time).all()

print("Split temporal validado sem sobreposição.")

Split temporal validado sem sobreposição.


## Ranking de popularidade

O ranking usa somente o histórico permitido em cada avaliação. A quantidade de interações define a ordem principal e a média das notas serve como desempate.

In [7]:
def build_popularity_ranking(history: pd.DataFrame) -> pd.DataFrame:
    ranking = (
        history.groupby("movie_id")
        .agg(
            interaction_count=("rating", "size"),
            rating_mean=("rating", "mean"),
        )
        .reset_index()
        .sort_values(
            ["interaction_count", "rating_mean", "movie_id"],
            ascending=[False, False, True],
            kind="stable",
        )
        .reset_index(drop=True)
    )
    return ranking


validation_ranking = build_popularity_ranking(train)
display(validation_ranking.head(10).merge(movies, on="movie_id", how="left"))

,movie_id,interaction_count,rating_mean,title,genres
0,356,325,4.1600,Forrest Gump (1994),Comedy|Drama|Romance|War
1,318,313,4.4313,"Shawshank Redemption, The (1994)",Crime|Drama
2,296,304,4.1924,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
3,2571,277,4.1895,"Matrix, The (1999)",Action|Sci-Fi|Thriller
4,593,276,4.1558,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
5,260,250,4.2280,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
6,480,236,3.7479,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
7,110,234,4.0235,Braveheart (1995),Action|Drama|War
8,589,221,3.9638,Terminator 2: Judgment Day (1991),Action|Sci-Fi
9,2959,218,4.2729,Fight Club (1999),Action|Crime|Drama|Thriller


## Recomendações Top-K

O recomendador percorre o ranking global e remove os filmes já vistos pelo usuário no histórico disponível.

In [8]:
def build_seen_items(history: pd.DataFrame) -> dict[int, set[int]]:
    return history.groupby("user_id")["movie_id"].agg(set).to_dict()


def recommend_popular(
    user_id: int,
    ranking: pd.DataFrame,
    seen_items: dict[int, set[int]],
    k: int = TOP_K,
) -> list[int]:
    seen = seen_items.get(user_id, set())
    candidates = ranking.loc[~ranking["movie_id"].isin(seen), "movie_id"]
    return candidates.head(k).astype(int).tolist()


def recommend_for_users(
    user_ids: pd.Series,
    ranking: pd.DataFrame,
    history: pd.DataFrame,
    k: int = TOP_K,
) -> dict[int, list[int]]:
    seen_items = build_seen_items(history)
    return {
        int(user_id): recommend_popular(int(user_id), ranking, seen_items, k)
        for user_id in user_ids.unique()
    }

## Métricas

Uma interação reservada é relevante quando `rating >= 4`. Como há somente um item por usuário em cada holdout, Recall@10 equivale a Hit Rate@10 entre os usuários avaliáveis. Precision@10 vale 0,1 quando o único item relevante aparece no Top-10.

In [9]:
def precision_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    hits = len(set(recommended[:k]) & relevant)
    return hits / k


def recall_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    if not relevant:
        return 0.0
    hits = len(set(recommended[:k]) & relevant)
    return hits / len(relevant)


def evaluate_popularity(
    history: pd.DataFrame,
    holdout: pd.DataFrame,
    k: int = TOP_K,
) -> tuple[pd.Series, pd.DataFrame, dict[int, list[int]]]:
    ranking = build_popularity_ranking(history)
    recommendations = recommend_for_users(holdout["user_id"], ranking, history, k)
    relevant_holdout = holdout.loc[holdout["rating"] >= RELEVANCE_THRESHOLD]
    relevant_by_user = relevant_holdout.groupby("user_id")["movie_id"].agg(set).to_dict()

    rows = []
    for user_id, relevant in relevant_by_user.items():
        recommended = recommendations[int(user_id)]
        rows.append(
            {
                "user_id": int(user_id),
                f"precision@{k}": precision_at_k(recommended, relevant, k),
                f"recall@{k}": recall_at_k(recommended, relevant, k),
                "hit": bool(set(recommended[:k]) & relevant),
            }
        )

    user_metrics = pd.DataFrame(rows)
    unique_recommended = {item for items in recommendations.values() for item in items[:k]}
    summary = pd.Series(
        {
            "holdout_users": holdout["user_id"].nunique(),
            "evaluable_users": len(relevant_by_user),
            f"precision@{k}": user_metrics[f"precision@{k}"].mean(),
            f"recall@{k}": user_metrics[f"recall@{k}"].mean(),
            f"hit_rate@{k}": user_metrics["hit"].mean(),
            f"catalog_coverage@{k}": len(unique_recommended) / len(ranking),
        },
        name="value",
    )
    return summary, user_metrics, recommendations

## Avaliação na validação

O ranking é construído somente com treino e tenta recuperar a penúltima interação relevante de cada usuário.

In [10]:
validation_summary, validation_user_metrics, validation_recommendations = (
    evaluate_popularity(train, validation, TOP_K)
)
validation_summary.to_frame()

,value
holdout_users,610.0000
evaluable_users,347.0000
precision@10,0.0043
recall@10,0.0432
hit_rate@10,0.0432
catalog_coverage@10,0.0125


## Avaliação final no teste

Para o teste, o histórico disponível inclui treino e validação. O ranking é reconstruído sem usar nenhuma interação de teste.

In [11]:
train_validation = (
    pd.concat([train, validation], ignore_index=True)
    .sort_values(["user_id", "rated_at", "movie_id"], kind="stable")
    .reset_index(drop=True)
)
test_summary, test_user_metrics, test_recommendations = evaluate_popularity(
    train_validation, test, TOP_K
)
test_summary.to_frame()

,value
holdout_users,610.0000
evaluable_users,363.0000
precision@10,0.0061
recall@10,0.0606
hit_rate@10,0.0606
catalog_coverage@10,0.0125


## Inspeção de recomendações

A inspeção qualitativa mostra as recomendações e indica qual filme foi reservado para teste.

In [12]:
def recommendation_details(
    user_id: int,
    recommendations: dict[int, list[int]],
    catalog: pd.DataFrame,
) -> pd.DataFrame:
    movie_ids = recommendations[user_id]
    ranked = pd.DataFrame(
        {"rank": range(1, len(movie_ids) + 1), "movie_id": movie_ids}
    )
    return ranked.merge(catalog, on="movie_id", how="left").sort_values("rank")


sample_user_ids = test_user_metrics["user_id"].head(3).tolist()
for user_id in sample_user_ids:
    held_out = test.loc[test["user_id"] == user_id, ["movie_id", "rating"]].merge(
        movies, on="movie_id", how="left"
    )
    print(f"\nUsuário {user_id} — item relevante reservado para teste")
    display(held_out)
    print("Top-10 recomendado")
    display(recommendation_details(user_id, test_recommendations, movies))


Usuário 1 — item relevante reservado para teste


,movie_id,rating,title,genres
0,2492,4.0000,20 Dates (1998),Comedy|Romance


Top-10 recomendado


,rank,movie_id,title,genres
0,1,318,"Shawshank Redemption, The (1994)",Crime|Drama
1,2,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
2,3,150,Apollo 13 (1995),Adventure|Drama|IMAX
3,4,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
4,5,858,"Godfather, The (1972)",Crime|Drama
5,6,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy
6,7,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy
7,8,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical
8,9,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery
9,10,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller



Usuário 2 — item relevante reservado para teste


...

,rank,movie_id,title,genres
0,1,356,Forrest Gump (1994),Comedy|Drama|Romance|War
1,2,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
2,3,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
3,4,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
4,5,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
5,6,110,Braveheart (1995),Action|Drama|War
6,7,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
7,8,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
8,9,2959,Fight Club (1999),Action|Crime|Drama|Thriller
9,10,527,Schindler's List (1993),Drama|War



Usuário 4 — item relevante reservado para teste


,movie_id,rating,title,genres
0,4246,4.0000,Bridget Jones's Diary (2001),Comedy|Drama|Romance


Top-10 recomendado


,rank,movie_id,title,genres
0,1,356,Forrest Gump (1994),Comedy|Drama|Romance|War
1,2,318,"Shawshank Redemption, The (1994)",Crime|Drama
2,3,110,Braveheart (1995),Action|Drama|War
3,4,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller
4,5,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
5,6,527,Schindler's List (1993),Drama|War
6,7,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
7,8,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
8,9,150,Apollo 13 (1995),Adventure|Drama|IMAX
9,10,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy


## Conclusões

- O baseline recomenda os mesmos filmes populares para todos os usuários, com diferenças causadas apenas pelo filtro de itens já vistos.
- O split temporal evita usar interações futuras na construção do ranking.
- Usuários cuja interação reservada tem nota menor que 4 não entram no cálculo de Precision/Recall, mas continuam contabilizados em `holdout_users`.
- A baixa cobertura é uma limitação esperada: um baseline de popularidade concentra recomendações em poucos títulos.
- Modelos personalizados posteriores deverão superar esse resultado, especialmente em Recall@10 e cobertura de catálogo.

## Próximo passo

Extrair split, recomendação e métricas para módulos em `src/`, cobri-los com testes e implementar um baseline personalizado com biases de usuário/item ou fatoração de matriz.